# Benchmark darija de Wakil, sur un GPU Google Colab

Ce notebook fait passer le benchmark de Wakil (`evals/benchmark`) aux modèles de `promptfooconfig.yaml`, sur un GPU
gratuit de Colab au lieu du PC. **Données 100 % fictives – projet de démonstration.**

**Avant de commencer**
1. Pousser sur GitHub les derniers changements (questions, code) : ce notebook clone le dépôt, pas le PC.
2. Dans Colab : *Exécution → Modifier le type d'exécution → GPU T4*.
3. Exécuter les cellules dans l'ordre (*Exécution → Tout exécuter*).

**Ce qu'il faut savoir**
- Aucun secret ni base de données : le benchmark n'a besoin que d'Ollama, des deux serveurs MCP (la liste des outils)
  et de promptfoo. Aucun outil n'est exécuté.
- Les modèles sont retéléchargés à chaque session (environ 14 Go, sur le réseau de Google).
- **Ne pas mélanger ces résultats avec ceux du PC** : un GPU calcule très légèrement autrement qu'un processeur.
  Tous les modèles comparés doivent passer sur la même machine. Les temps de réponse sont ceux d'un GPU T4.

## 1. Réglages

In [1]:
DEPOT = "https://github.com/Oumaima-Lg/wakil.git"
BRANCHE = "main"
FILTRE = ""  # "" : tous les modèles de promptfooconfig.yaml ; « qwen » : seulement ceux dont le nom contient « qwen »

## 2. Le GPU est-il là ?

In [2]:
import subprocess

sortie = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if sortie.returncode != 0:
    raise SystemExit("Pas de GPU : Exécution → Modifier le type d'exécution → GPU T4, puis relancer.")
print(sortie.stdout)

Sun Nov  1 14:18:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Le dépôt

In [3]:
%cd /content
!rm -rf wakil && git clone --quiet --depth 1 --branch {BRANCHE} {DEPOT} wakil
!cd wakil && git log --oneline -1

/content
07a9dfa (grafted, HEAD -> main, origin/main, origin/HEAD) Benchmark : 39 questions d'Oumaima, conversations annotées 5 et 6, fiches de la FAQ à jour, notebook Colab pour mesurer sur GPU


## 4. Ollama et les modèles

Les modèles à télécharger sont lus dans `promptfooconfig.yaml` : la même liste que celle du benchmark.

In [4]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import time
import urllib.request

ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        break
    except OSError:
        time.sleep(1)
else:
    raise SystemExit("Ollama ne démarre pas : voir /content/ollama.log")
print("Ollama répond.")

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama répond.


In [5]:
import yaml

config = yaml.safe_load(open("/content/wakil/evals/benchmark/promptfooconfig.yaml"))
modeles = [f["config"]["modele"] for f in config["providers"] if FILTRE in f["label"]]
print("Modèles :", modeles)
for modele in modeles:
    print(f"\n=== {modele}")
    subprocess.run(["ollama", "pull", modele], check=True, stdout=subprocess.DEVNULL)
!ollama list

Modèles : ['qwen3:4b-instruct', 'qwen3:8b', 'hf.co/mradermacher/Atlas-Chat-9B-GGUF:Q4_K_M']

=== qwen3:4b-instruct

=== qwen3:8b

=== hf.co/mradermacher/Atlas-Chat-9B-GGUF:Q4_K_M
NAME                                            ID              SIZE      MODIFIED               
hf.co/mradermacher/Atlas-Chat-9B-GGUF:Q4_K_M    e03413840ac0    5.8 GB    Less than a second ago    
qwen3:8b                                        500a1f067a9f    5.2 GB    About a minute ago        
qwen3:4b-instruct                               0edcdef34593    2.5 GB    2 minutes ago             


## 5. Les dépendances

uv installe Python 3.14 et les paquets des trois services utiles ; Node 22 sert à promptfoo.

In [6]:
!pip install -q uv
for service in ("agent-api", "mcp-tools", "rag-faq"):
    print("===", service)
    subprocess.run(["uv", "sync", "--quiet"], cwd=f"/content/wakil/apps/{service}", check=True)

!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1 && apt-get -qq install -y nodejs > /dev/null
!node --version
!cd /content/wakil/evals && npm ci --silent --no-audit --no-fund

=== agent-api
=== mcp-tools
=== rag-faq
v22.23.4


## 6. Les deux serveurs MCP

Ils annoncent les outils de Wakil et leurs descriptions (sans base de données : aucun outil n'est exécuté).

In [7]:
import socket

serveurs = {}
for service, port in (("mcp-tools", 8002), ("rag-faq", 8004)):
    serveurs[service] = subprocess.Popen(
        ["uv", "run", "uvicorn", "app.serveur:app", "--port", str(port)],
        cwd=f"/content/wakil/apps/{service}",
        stdout=open(f"/content/{service}.log", "w"),
        stderr=subprocess.STDOUT,
    )
    for _ in range(60):
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", port)) == 0:
                break
        time.sleep(1)
    else:
        raise SystemExit(f"{service} ne démarre pas : voir /content/{service}.log")
    print(service, "répond sur le port", port)

mcp-tools répond sur le port 8002
rag-faq répond sur le port 8004


## 7. Vérification avant la mesure

Les outils que le LLM verra, annoncés par les serveurs MCP : ils doivent tous être là.

In [8]:
!cd /content/wakil/evals/benchmark && ../../apps/agent-api/.venv/bin/python -c "import fournisseur; print(sorted(d['function']['name'] for d in fournisseur._descriptions()))"

<frozen site>:84: RuntimeWarning: Unexpected value in sys.prefix, expected /content/wakil/apps/agent-api/.venv, got /content/wakil/evals/benchmark/../../apps/agent-api/.venv
<frozen site>:84: RuntimeWarning: Unexpected value in sys.exec_prefix, expected /content/wakil/apps/agent-api/.venv, got /content/wakil/evals/benchmark/../../apps/agent-api/.venv
['analyser_depenses', 'bloquer_carte', 'consulter_cartes', 'consulter_plafond', 'consulter_solde', 'faire_virement', 'lister_beneficiaires', 'lister_factures', 'lister_transactions', 'payer_facture', 'question_faq']


## 8. La mesure

Un modèle à la fois, puis le rapport (`resultats.md`).

In [9]:
!cd /content/wakil/evals && npm run benchmark -- {FILTRE}


> benchmark
> ../apps/agent-api/.venv/bin/python benchmark/lancer.py

<frozen site>:84: RuntimeWarning: Unexpected value in sys.prefix, expected /content/wakil/apps/agent-api/.venv, got /content/wakil/evals/../apps/agent-api/.venv
<frozen site>:84: RuntimeWarning: Unexpected value in sys.exec_prefix, expected /content/wakil/apps/agent-api/.venv, got /content/wakil/evals/../apps/agent-api/.venv

=== qwen3:4b-instruct
⠙Cache is disabled.
Starting evaluation eval-MxW-2026-11-01T14:26:28
Running 170 test cases (up to 1 at a time)...

┌─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ question    │ avant       │ categorie   │ ecriture    │ outil       │ arguments   │ paquet      │ source      │ [qwen3:4b-… │
│             │             │             │             │             │             │             │             │ {{question… │
├─────────────┼─────────────┼─────────────┼─────────────┼─────────────┼────────

## 9. Les résultats

In [10]:
from IPython.display import Markdown, display

display(Markdown(open("/content/wakil/evals/benchmark/resultats.md", encoding="utf-8").read()))

# Benchmark darija de Wakil : résultats mesurés

Généré le 01/11/2026 par `evals/benchmark/rapport.py`, à partir de promptfoo. Données 100 % fictives.

**Ce qui est mesuré** : l'étape 1 de Wakil, par le vrai chemin de l'agent (même prompt, mêmes outils MCP, même historique). Une réponse est réussie si le LLM choisit le bon outil, avec les paramètres attendus, sans rien inventer. Aucun outil n'est exécuté.

**Les questions** : 113 jamais vues (paquet « test », qui donne la note) et 57 déjà utilisées pour régler Wakil (paquet « mise au point », score optimiste, donné à part).

## Classement

| Modèle | Bon outil (test) | Outil et paramètres justes (test) | Mise au point (outil et paramètres) | Temps médian | Temps p95 | Jetons par question | JSON illisible (mode json) |
|---|---|---|---|---|---|---|---|
| `atlas-chat:9b` | 85 % (96/113) | **81 % (91/113)** | 56 % (32/57) | 1.6 s | 2.3 s | 1073 | 0 |
| `qwen3:4b-instruct` | 83 % (94/113) | **78 % (88/113)** | 44 % (25/57) | 0.4 s | 1.7 s | 1721 | 0 |
| `qwen3:8b` | 80 % (90/113) | **73 % (82/113)** | 26 % (15/57) | 0.7 s | 2.3 s | 1528 | 0 |

## Comparaison question par question (paquet test)

Sur les questions où deux modèles ne sont pas d'accord : si p est petit (moins de 0,05), l'écart n'est probablement pas dû au hasard.

| Modèle A | Modèle B | Seul A réussit | Seul B réussit | p (McNemar) |
|---|---|---|---|---|
| `atlas-chat:9b` | `qwen3:4b-instruct` | 14 | 11 | 0.690 |
| `atlas-chat:9b` | `qwen3:8b` | 17 | 8 | 0.108 |
| `qwen3:4b-instruct` | `qwen3:8b` | 11 | 5 | 0.210 |

## Par écriture (tous paquets)

| Modèle | arabe | latin | melange | francais | anglais | arabe_standard |
|---|---|---|---|---|---|---|
| `atlas-chat:9b` | 79 % (26/33) | 67 % (38/57) | 68 % (27/40) | 80 % (16/20) | 80 % (8/10) | 80 % (8/10) |
| `qwen3:4b-instruct` | 52 % (17/33) | 56 % (32/57) | 72 % (29/40) | 75 % (15/20) | 100 % (10/10) | 100 % (10/10) |
| `qwen3:8b` | 52 % (17/33) | 46 % (26/57) | 58 % (23/40) | 65 % (13/20) | 90 % (9/10) | 90 % (9/10) |

## Par catégorie (tous paquets)

| Modèle | beneficiaires | cartes | depenses | factures | faq | floue | hors_sujet | non_disponible | operations | paiement | plafond | plusieurs | solde | suite | virement |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| `atlas-chat:9b` | 100 % (7/7) | 83 % (5/6) | 100 % (8/8) | 100 % (8/8) | 56 % (31/55) | 75 % (3/4) | 18 % (2/11) | 0 % (0/3) | 60 % (6/10) | 87 % (13/15) | 100 % (4/4) | 100 % (6/6) | 100 % (14/14) | 60 % (3/5) | 93 % (13/14) |
| `qwen3:4b-instruct` | 100 % (7/7) | 100 % (6/6) | 88 % (7/8) | 100 % (8/8) | 38 % (21/55) | 0 % (0/4) | 82 % (9/11) | 67 % (2/3) | 70 % (7/10) | 73 % (11/15) | 100 % (4/4) | 100 % (6/6) | 93 % (13/14) | 40 % (2/5) | 71 % (10/14) |
| `qwen3:8b` | 100 % (7/7) | 100 % (6/6) | 88 % (7/8) | 88 % (7/8) | 11 % (6/55) | 0 % (0/4) | 91 % (10/11) | 67 % (2/3) | 10 % (1/10) | 73 % (11/15) | 100 % (4/4) | 100 % (6/6) | 93 % (13/14) | 60 % (3/5) | 100 % (14/14) |

## Échecs du paquet test

| Modèle | Question | Écriture | Catégorie | Pourquoi |
|---|---|---|---|---|
| `atlas-chat:9b` | achno drt b lflous f had l'ayam ? | latin | operations | outil analyser_depenses au lieu de lister_transactions |
| `atlas-chat:9b` | شنو خلصت هاد الأيام؟ | arabe | operations | outil analyser_depenses au lieu de lister_transactions |
| `atlas-chat:9b` | خلص ليا كاع الفواتير | arabe | paiement | fournisseur = 'tout' au lieu de 'toutes' |
| `atlas-chat:9b` | ila sifet l chi wa7ed daba, kaywsel f nefs nhar ? | latin | faq | outil faire_virement au lieu de question_faq |
| `atlas-chat:9b` | bghit nkhallas wa7d l facture | latin | floue | fournisseur inventé : 'eau_electricite' |
| `atlas-chat:9b` | شنو هي العاصمة ديال اليابان؟ | arabe | hors_sujet | outil question_faq au lieu de None |
| `atlas-chat:9b` | 3tini chi nokta | latin | hors_sujet | outil question_faq au lieu de None |
| `atlas-chat:9b` | écris-moi un email l mudir dyali | melange | hors_sujet | outil question_faq au lieu de None |
| `atlas-chat:9b` | w 3awed 100 dh l nefs l wa7ed | latin | suite | beneficiaire = None au lieu de 'Fatima' ; montant = None au lieu de 100 |
| `atlas-chat:9b` | w chhal b9a liya ? | melange | suite | outil analyser_depenses au lieu de consulter_solde |
| `atlas-chat:9b` | bghit ncommandi chéquier jdid | melange | non_disponible | outil question_faq au lieu de None |
| `atlas-chat:9b` | بغيت نبدل كلمة السر ديال التطبيق | arabe | non_disponible | outil question_faq au lieu de None |
| `atlas-chat:9b` | Show me my last transactions | anglais | operations | mois inventé : 'ce_mois' |
| `atlas-chat:9b` | أرني آخر العمليات في حسابي | arabe_standard | operations | mois inventé : 'ce_mois' |
| `atlas-chat:9b` | اكتب لي قصيدة عن البحر | arabe_standard | hors_sujet | outil question_faq au lieu de None |
| `atlas-chat:9b` | chkoun rbe7 l'match dyal lbare7 ? | latin | hors_sujet | outil question_faq au lieu de None |
| `atlas-chat:9b` | Ma carte est deja bloquée ? | francais | cartes | outil question_faq au lieu de consulter_cartes |
| `atlas-chat:9b` | dibloque liya la carte dyali | melange | faq | outil bloquer_carte au lieu de question_faq |
| `atlas-chat:9b` | Merci de débloquer ma carte | francais | faq | outil None au lieu de question_faq |
| `atlas-chat:9b` | zide liya benifciaire jdide | latin | faq | outil lister_beneficiaires au lieu de question_faq |
| `atlas-chat:9b` | bgjite nzide benifiaire | latin | faq | outil lister_beneficiaires au lieu de question_faq |
| `atlas-chat:9b` | je veux ajouter un bénéficiare | francais | faq | outil lister_beneficiaires au lieu de question_faq |
| `qwen3:4b-instruct` | achno drt b lflous f had l'ayam ? | latin | operations | outil question_faq au lieu de lister_transactions |
| `qwen3:4b-instruct` | شنو خلصت هاد الأيام؟ | arabe | operations | outil lister_factures au lieu de lister_transactions |
| `qwen3:4b-instruct` | dir liya paiement dyal facture lma | latin | paiement | outil lister_factures au lieu de payer_facture ; fournisseur = None au lieu de 'eau_electricite' |
| `qwen3:4b-instruct` | khalless ga3 les factures li 3liya | melange | paiement | outil lister_factures au lieu de payer_facture ; fournisseur = None au lieu de 'toutes' |
| `qwen3:4b-instruct` | خلص ليا كاع الفواتير | arabe | paiement | outil lister_factures au lieu de payer_facture ; fournisseur = None au lieu de 'toutes' |
| `qwen3:4b-instruct` | صيفط لفاطمة 200 درهم | arabe | virement | beneficiaire = 'fatima' au lieu de 'فاطمة' |
| `qwen3:4b-instruct` | wach Wakil kaykhdem nhar l3id ? | latin | faq | outil None au lieu de question_faq |
| `qwen3:4b-instruct` | واش هاد البنك ديال بصح ولا غير تجربة؟ | arabe | faq | outil None au lieu de question_faq |
| `qwen3:4b-instruct` | بغيت نخلص فاتورة | arabe | floue | outil lister_factures au lieu de payer_facture |
| `qwen3:4b-instruct` | bghit nkhallas wa7d l facture | latin | floue | outil lister_factures au lieu de payer_facture |
| `qwen3:4b-instruct` | w 3awed 100 dh l nefs l wa7ed | latin | suite | beneficiaire = 'nef' au lieu de 'Fatima' |
| `qwen3:4b-instruct` | w chhal b9a liya ? | melange | suite | outil analyser_depenses au lieu de consulter_solde ; categorie inventé : 'restaurant' |
| `qwen3:4b-instruct` | شحال صرفت الشهر اللي فات؟ | arabe | depenses | periode = 'ce_mois' au lieu de 'mois_dernier' |
| `qwen3:4b-instruct` | bghit ndir virement | latin | floue | outil None au lieu de faire_virement |
| `qwen3:4b-instruct` | Bghite nsifte 2000 DH l Fatima, wach kayn solde kafi ? | latin | solde | outil lister_beneficiaires au lieu de consulter_solde |
| `qwen3:4b-instruct` | dibloque liya la carte dyali | melange | faq | outil bloquer_carte au lieu de question_faq ; carte inventé : '1234' |
| `qwen3:4b-instruct` | Merci de débloquer ma carte | francais | faq | outil None au lieu de question_faq |
| `qwen3:4b-instruct` | chno kadire f 7yatke ? | latin | faq | outil None au lieu de question_faq |
| `qwen3:4b-instruct` | qu'est ce que tu fais dans ta vie ? | francais | faq | outil None au lieu de question_faq |
| `qwen3:4b-instruct` | zide liya benifciaire jdide | latin | faq | outil lister_beneficiaires au lieu de question_faq |
| `qwen3:4b-instruct` | bgjite nzide benifiaire | latin | faq | outil lister_beneficiaires au lieu de question_faq |
| `qwen3:4b-instruct` | je veux ajouter un bénéficiare | francais | faq | outil None au lieu de question_faq |
| `qwen3:4b-instruct` | bghite nsifte 2000 dh liya ana fl compte dyali. | latin | virement | beneficiaire inventé : 'Ali' |
| `qwen3:4b-instruct` | je veux envoyer 2000 dh à moi-même dans mon compte. | francais | virement | beneficiaire inventé : 'self' |
| `qwen3:4b-instruct` | sf mzn, daba sifte liha 456.78 dh | latin | suite | beneficiaire inventé : 'Ahmed' |
| `qwen3:8b` | ورّيني آخر العمليات اللي دازو فالحساب | arabe | operations | mois inventé : 12 |
| `qwen3:8b` | achno drt b lflous f had l'ayam ? | latin | operations | outil None au lieu de lister_transactions |
| `qwen3:8b` | affichili les dernières dépenses dyali | melange | operations | mois inventé : 1 |
| `qwen3:8b` | شنو خلصت هاد الأيام؟ | arabe | operations | outil analyser_depenses au lieu de lister_transactions ; categorie inventé : 'factures' |
| `qwen3:8b` | chouf liya l7arakat dyal l compte | latin | operations | outil consulter_cartes au lieu de lister_transactions |
| `qwen3:8b` | ach men factures mazal 3liya ? | latin | factures | outil None au lieu de lister_factures |
| `qwen3:8b` | dir liya paiement dyal facture lma | latin | paiement | outil lister_factures au lieu de payer_facture ; fournisseur = None au lieu de 'eau_electricite' |
| `qwen3:8b` | khalless ga3 les factures li 3liya | melange | paiement | outil lister_factures au lieu de payer_facture ; fournisseur = None au lieu de 'toutes' |
| `qwen3:8b` | خلص ليا كاع الفواتير | arabe | paiement | outil lister_factures au lieu de payer_facture ; fournisseur = None au lieu de 'toutes' |
| `qwen3:8b` | wach Wakil kaykhdem nhar l3id ? | latin | faq | outil None au lieu de question_faq |
| `qwen3:8b` | si quelqu'un m'appelle w ytleb code SMS, chno ndir ? | melange | faq | outil None au lieu de question_faq |
| `qwen3:8b` | واش هاد البنك ديال بصح ولا غير تجربة؟ | arabe | faq | outil None au lieu de question_faq |
| `qwen3:8b` | ila sifet l chi wa7ed daba, kaywsel f nefs nhar ? | latin | faq | outil faire_virement au lieu de question_faq ; montant inventé : 500 |
| `qwen3:8b` | Est-ce que je peux changer mon plafond de virement ? | francais | faq | outil None au lieu de question_faq |
| `qwen3:8b` | بغيت نخلص فاتورة | arabe | floue | outil lister_factures au lieu de payer_facture |
| `qwen3:8b` | bghit nkhallas wa7d l facture | latin | floue | outil lister_factures au lieu de payer_facture |
| `qwen3:8b` | w 3awed 100 dh l nefs l wa7ed | latin | suite | beneficiaire = 'nefs' au lieu de 'Fatima' |
| `qwen3:8b` | w chhal b9a liya ? | melange | suite | outil analyser_depenses au lieu de consulter_solde ; categorie inventé : 'retraits' |
| `qwen3:8b` | chhal srefti had chhar ? | latin | depenses | categorie inventé : 'toutes' |
| `qwen3:8b` | Show me my last transactions | anglais | operations | mois inventé : 12 |
| `qwen3:8b` | أرني آخر العمليات في حسابي | arabe_standard | operations | mois inventé : 12 |
| `qwen3:8b` | bghit nchouf akhir les opérations dyali | melange | operations | mois inventé : 12 |
| `qwen3:8b` | bghit ndir virement | latin | floue | beneficiaire inventé : 'nom du bénéficiaire' ; montant inventé : 0 |
| `qwen3:8b` | Bghite nsifte 2000 DH l Fatima, wach kayn solde kafi ? | latin | solde | outil faire_virement au lieu de consulter_solde |
| `qwen3:8b` | dibloque liya la carte dyali | melange | faq | outil bloquer_carte au lieu de question_faq ; carte inventé : '4321' |
| `qwen3:8b` | Merci de débloquer ma carte | francais | faq | outil None au lieu de question_faq |
| `qwen3:8b` | chno kadire f 7yatke ? | latin | faq | outil None au lieu de question_faq |
| `qwen3:8b` | qu'est ce que tu fais dans ta vie ? | francais | faq | outil None au lieu de question_faq |
| `qwen3:8b` | zide liya benifciaire jdide | latin | faq | outil lister_beneficiaires au lieu de question_faq |
| `qwen3:8b` | bgjite nzide benifiaire | latin | faq | outil lister_beneficiaires au lieu de question_faq |
| `qwen3:8b` | je veux ajouter un bénéficiare | francais | faq | outil None au lieu de question_faq |


In [11]:
import shutil

from google.colab import files

shutil.copy("/content/wakil/evals/benchmark/resultats.md", "/content/wakil/evals/benchmark/resultats/resultats.md")
archive = shutil.make_archive("/content/resultats-benchmark", "zip", "/content/wakil/evals/benchmark/resultats")
files.download(archive)  # resultats.md et les réponses brutes de chaque modèle (JSON)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>